# Tutorial 6: BME with Soft Probabilistic Data

**Corresponds to MATLAB `BMEPROBALIBtutorial.m`**

This is the flagship tutorial demonstrating the full BME workflow:
1. Generate synthetic hard + soft data on a 2-D domain
2. Compute BME moments (mean, variance) on an estimation grid
3. Extract the full posterior PDF + confidence intervals at a point
4. Compare BME vs kriging maps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
%matplotlib inline

from pybme import bme_predict, SoftPDF, eval_cov, build_cov_matrix

## 1. Simulate a Gaussian Random Field

We use Cholesky decomposition of the covariance matrix to draw from the random field, then split locations into hard and soft data.

In [ ]:
def simulate_grf(coords, model, params, mean, seed=42):
    """Simulate from a Gaussian random field via Cholesky."""
    K = build_cov_matrix(coords, coords, model, params)
    K += np.eye(len(coords)) * 1e-10
    L = np.linalg.cholesky(K)
    rng = np.random.default_rng(seed)
    return mean + L @ rng.standard_normal(len(coords))

rng = np.random.default_rng(42)

# Candidate locations on a 5×6 grid
gx, gy = np.meshgrid(np.arange(0, 9, 2), np.arange(0, 11, 2))
candidates = np.column_stack([gx.ravel(), gy.ravel()])  # 30 pts
idx = rng.permutation(len(candidates))

n_hard, n_soft = 10, 11
ch = candidates[idx[:n_hard]]
cs = candidates[idx[n_hard:n_hard + n_soft]]

model, params, mean_val = 'exponential', [1.0, 5.0], 3.0
z_all = simulate_grf(candidates, model, params, mean_val, seed=42)
zh = z_all[idx[:n_hard]]
z_soft_true = z_all[idx[n_hard:n_hard + n_soft]]

print(f"Hard data:  {n_hard} points")
print(f"Soft data:  {n_soft} points")

## 2. Construct Soft PDFs

Each soft datum is represented by a **trapezoidal PDF** — a flat (uniform-ish) distribution centered on the true value. This mimics e.g. expert knowledge or censored data.

In [ ]:
soft_pdfs = []
for zt in z_soft_true:
    width = rng.uniform(0.5, 1.5)
    lo, hi = zt - width, zt + width
    z_grid = np.linspace(lo - 0.3, hi + 0.3, 25)
    pdf_vals = np.where((z_grid >= lo) & (z_grid <= hi), 1.0, 0.0)
    soft_pdfs.append(SoftPDF.from_linear(z_grid, pdf_vals))

# Quick look at the first 4 soft PDFs
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, i in zip(axes.ravel(), range(4)):
    sp = soft_pdfs[i]
    z = np.linspace(*sp.support, 200)
    ax.plot(z, sp.evaluate(z), 'b-', lw=1.5)
    ax.fill_between(z, sp.evaluate(z), alpha=0.2)
    ax.set_title(f'Soft point {i+1} at ({cs[i,0]:.0f}, {cs[i,1]:.0f})')
    ax.set_xlabel('z')
    ax.set_ylabel('f(z)')
    ax.grid(True, alpha=0.3)
fig.suptitle('Soft PDFs at Selected Points')
fig.tight_layout()
plt.show()

## 3. BME Estimation on a Grid

Run `bme_predict` over a 9×11 estimation grid. This returns both the BME posterior moments **and** the kriging-only moments for comparison.

In [ ]:
gx_e, gy_e = np.meshgrid(np.arange(0, 9, 1), np.arange(0, 11, 1))
ck = np.column_stack([gx_e.ravel(), gy_e.ravel()])
print(f"Estimation grid: {ck.shape[0]} points ({gx_e.shape[1]}×{gx_e.shape[0]})")

results = bme_predict(
    ck, ch, zh, cs, soft_pdfs,
    model=model, params=params,
    nhmax=10, nsmax=2, dmax=100.0,
    order=0, n_grid=150, ci_prob=0.95,
)

zk      = np.array([r.mean for r in results])
vk      = np.array([r.variance for r in results])
zk_krig = np.array([r.kriging_mean for r in results])
vk_krig = np.array([r.kriging_var for r in results])

print(f"\nBME  — mean estimate: {zk.mean():.3f},  mean variance: {vk.mean():.3f}")
print(f"Krig — mean estimate: {zk_krig.mean():.3f},  mean variance: {vk_krig.mean():.3f}")
print(f"\nVariance reduction: {(1 - vk.mean()/vk_krig.mean())*100:.1f}%")

## 4. Mean Estimate Maps: BME vs Kriging

In [ ]:
nx, ny = gx_e.shape[1], gx_e.shape[0]

fig, ax = plt.subplots(figsize=(6, 7))
ax.scatter(ch[:, 0], ch[:, 1], c=zh, cmap='hot', marker='s', s=80,
           edgecolors='k', zorder=5, label='hard')
soft_means = [sp.moments()[0] for sp in soft_pdfs]
ax.scatter(cs[:, 0], cs[:, 1], c=soft_means, cmap='hot', marker='o',
           s=60, edgecolors='b', linewidths=1.5, zorder=4, label='soft')
ax.set_title('Hard & Soft Data Locations')
ax.legend()
ax.set_aspect('equal')
fig.tight_layout()
plt.show()

In [ ]:
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

im1 = ax1.pcolormesh(gx_e, gy_e, zk.reshape(ny, nx), cmap='hot', shading='auto')
ax1.set_title('BME Mean Estimate')
ax1.set_aspect('equal')
plt.colorbar(im1, ax=ax1)

im2 = ax2.pcolormesh(gx_e, gy_e, zk_krig.reshape(ny, nx), cmap='hot', shading='auto')
ax2.set_title('Kriging Mean Estimate')
ax2.set_aspect('equal')
plt.colorbar(im2, ax=ax2)

fig2.tight_layout()
plt.show()

## 5. Error Variance Maps: BME vs Kriging

In [ ]:
fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

im1 = ax1.pcolormesh(gx_e, gy_e, vk.reshape(ny, nx), cmap='YlOrRd', shading='auto')
ax1.set_title('BME Error Variance')
ax1.set_aspect('equal')
plt.colorbar(im1, ax=ax1)

im2 = ax2.pcolormesh(gx_e, gy_e, vk_krig.reshape(ny, nx), cmap='YlOrRd', shading='auto')
ax2.set_title('Kriging Error Variance')
ax2.set_aspect('equal')
plt.colorbar(im2, ax=ax2)

fig3.tight_layout()
plt.show()

## 6. Full Posterior PDF at a Single Point

Extract the complete posterior distribution at point (5, 5) with confidence intervals at 68%, 90%, and 99% levels.

In [ ]:
ck_pt = np.array([[5.0, 5.0]])

# Get posterior at multiple CI levels
res_68 = bme_predict(ck_pt, ch, zh, cs, soft_pdfs,
                     model=model, params=params,
                     nhmax=10, nsmax=2, dmax=100.0,
                     order=0, n_grid=200, ci_prob=0.68)[0]

res_90 = bme_predict(ck_pt, ch, zh, cs, soft_pdfs,
                     model=model, params=params,
                     nhmax=10, nsmax=2, dmax=100.0,
                     order=0, n_grid=200, ci_prob=0.90)[0]

res_99 = bme_predict(ck_pt, ch, zh, cs, soft_pdfs,
                     model=model, params=params,
                     nhmax=10, nsmax=2, dmax=100.0,
                     order=0, n_grid=200, ci_prob=0.99)[0]

print(f"Posterior at (5, 5):")
print(f"  mode     = {res_68.mode:.3f}")
print(f"  mean     = {res_68.mean:.3f}")
print(f"  variance = {res_68.variance:.3f}")
print(f"  68% CI   = [{res_68.ci_lower:.2f}, {res_68.ci_upper:.2f}]")
print(f"  90% CI   = [{res_90.ci_lower:.2f}, {res_90.ci_upper:.2f}]")
print(f"  99% CI   = [{res_99.ci_lower:.2f}, {res_99.ci_upper:.2f}]")

In [ ]:
fig4, ax4 = plt.subplots(figsize=(9, 5))

ax4.plot(res_68.z_grid, res_68.pdf, 'b-', lw=2, label='posterior PDF')

# CI shading (widest first so narrower bands are on top)
for r, alpha, lab in [(res_99, 0.08, '99%'),
                       (res_90, 0.15, '90%'),
                       (res_68, 0.25, '68%')]:
    mask = (res_68.z_grid >= r.ci_lower) & (res_68.z_grid <= r.ci_upper)
    ax4.fill_between(res_68.z_grid, 0, res_68.pdf, where=mask,
                     alpha=alpha, color='blue', label=f'{lab} CI')

ax4.axvline(res_68.mode, color='r', ls='--', lw=1,
            label=f'mode = {res_68.mode:.2f}')
ax4.set_xlabel('z')
ax4.set_ylabel('f(z)')
ax4.set_title('BME Posterior PDF at (5, 5)')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
fig4.tight_layout()
plt.show()